# 0825_lsw_009_xgboost_hyperparameter_search

003~008에서 건드린 건 전부 "데이터를 어떻게 바꿀지"(리샘플링/라벨정제/가중치)나 "임계값을
어떻게 정할지"였다. **XGBoost 모델 자체의 하이퍼파라미터는 한 번도 튜닝하지 않았다** — 계속
`max_depth`/`min_child_weight`/`gamma`/`subsample`/`colsample_bytree` 전부 기본값이었다.

지금까지 반복 관찰된 증상(임계값이 1e-6~1e-7까지 내려감, 표본 작은 유형에서 결과가 크게
흔들림, PR-AUC는 좋아지는데 운영지표는 나빠짐)은 전형적인 과적합/정규화 부족 신호라, 정규화
계열 하이퍼파라미터를 가볍게 탐색해본다.

- **Phase A**: 리샘플링 없이(baseline 데이터) 하이퍼파라미터만 바꿔서 baseline과 비교 —
  하이퍼파라미터 자체의 순수 효과를 본다.
- **Phase B**: Phase A에서 이긴 설정을 **각 유형의 현재 최고 기법(리샘플링/라벨정제) 위에
  얹어서** 현재 최고 대비 더 개선되는지 확인한다(최근 합의한 "현재 최고 유지 + 개선분 탐색"
  방식).

탐색 범위는 넓은 그리드서치가 아니라 무작위 20개 조합(random search)으로 가볍게 한다 —
데이터가 작아서(가장 큰 type3도 8만 행) fit 자체는 빠르다.


## 1. 설정과 라이브러리

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from imblearn.over_sampling import ADASYN, SMOTE
from imblearn.under_sampling import RandomUnderSampler
from sklearn.metrics import average_precision_score, confusion_matrix, roc_auc_score
from xgboost import XGBClassifier

EXPERIMENT_ID = "0825_lsw_009_xgboost_hyperparameter_search"
RANDOM_STATE = 42
DATA_PATH = Path("../data/raw/dataset.csv")
TARGET = "class"
TIME_COLUMN = "timestamp"
RECORD_ID = "record_id"

COST_SCENARIOS = {"1:10": (1, 10), "1:100": (1, 100)}

assert DATA_PATH.exists(), f"파일을 찾을 수 없습니다: {DATA_PATH.resolve()}"
print("experiment:", EXPERIMENT_ID)


experiment: 0825_lsw_009_xgboost_hyperparameter_search


## 2. 데이터 로딩·전처리 (003~008과 동일)

In [2]:
raw_df = pd.read_csv(DATA_PATH, low_memory=False)
source_index_column = raw_df.columns[0]
if source_index_column.startswith("Unnamed:"):
    raw_df = raw_df.rename(columns={source_index_column: RECORD_ID})
elif source_index_column != RECORD_ID:
    raise ValueError(f"예상하지 못한 첫 번째 컬럼: {source_index_column}")
assert raw_df[RECORD_ID].is_unique, "record_id가 고유하지 않습니다."

dedup_columns = [c for c in raw_df.columns if c not in {RECORD_ID, TIME_COLUMN}]
duplicate_mask = raw_df.duplicated(subset=dedup_columns, keep="first")
clean_df = raw_df.loc[~duplicate_mask].copy().reset_index(drop=True)

clean_df[TIME_COLUMN] = pd.to_datetime(clean_df[TIME_COLUMN], errors="raise", utc=True)
clean_df = clean_df.sort_values([TIME_COLUMN, RECORD_ID], kind="stable").reset_index(drop=True)

feature_columns_all = [c for c in clean_df.columns if c not in {RECORD_ID, TIME_COLUMN, TARGET}]

timestamps = clean_df[TIME_COLUMN]
timestamp_group_sizes = timestamps.value_counts(sort=False).sort_index()
cumulative_rows = timestamp_group_sizes.cumsum().to_numpy()
train_end_time = timestamp_group_sizes.index[int(np.searchsorted(cumulative_rows, len(clean_df) * 0.60, side="left"))]
valid_end_time = timestamp_group_sizes.index[int(np.searchsorted(cumulative_rows, len(clean_df) * 0.80, side="left"))]

train_mask = timestamps <= train_end_time
valid_mask = (timestamps > train_end_time) & (timestamps <= valid_end_time)
test_mask = timestamps > valid_end_time


## 3. 평가 함수 (003~008과 동일)

In [3]:
def slip_rate(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    actual_positive = y_true == 1
    if actual_positive.sum() == 0:
        return 0.0
    fn = ((y_pred == 0) & actual_positive).sum()
    return fn / actual_positive.sum()


def volume_reduction(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    actual_negative = y_true == 0
    if actual_negative.sum() == 0:
        return 0.0
    tn = ((y_pred == 0) & actual_negative).sum()
    return tn / actual_negative.sum()


def total_cost(y_true, y_pred, cost_fp, cost_fn):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    fn = ((y_pred == 0) & (y_true == 1)).sum()
    fp = ((y_pred == 1) & (y_true == 0)).sum()
    return fn * cost_fn + fp * cost_fp


def select_threshold(y_val, proba_val, max_slip_rate=0.01):
    candidates = np.sort(np.unique(proba_val))[::-1]
    for t in candidates:
        y_pred = (proba_val >= t).astype(int)
        if slip_rate(y_val, y_pred) <= max_slip_rate:
            return float(t)
    return 0.0


def evaluate_at_threshold(y_true, proba, threshold):
    y_pred = (proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    result = {
        "threshold": threshold,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "slip_rate": slip_rate(y_true, y_pred),
        "volume_reduction": volume_reduction(y_true, y_pred),
        "pr_auc": average_precision_score(y_true, proba),
        "roc_auc": roc_auc_score(y_true, proba) if len(np.unique(y_true)) > 1 else float("nan"),
    }
    for name, (cost_fp, cost_fn) in COST_SCENARIOS.items():
        result[f"total_cost_{name}"] = total_cost(y_true, y_pred, cost_fp, cost_fn)
    return result


def get_non_constant_columns(candidate_columns, train_frame):
    nunique = train_frame[candidate_columns].nunique()
    return nunique[nunique > 1].index.tolist()


def resample_train(technique, X_train, y_train, n_pos):
    k_neighbors = max(1, min(5, n_pos - 1))
    if technique == "smote":
        return SMOTE(random_state=RANDOM_STATE, k_neighbors=k_neighbors).fit_resample(X_train, y_train)
    if technique == "adasyn":
        try:
            return ADASYN(random_state=RANDOM_STATE, n_neighbors=k_neighbors).fit_resample(X_train, y_train)
        except ValueError:
            return X_train, y_train
    if technique == "undersample":
        return RandomUnderSampler(random_state=RANDOM_STATE).fit_resample(X_train, y_train)
    return X_train, y_train


## 4. 검사유형별 subset 및 baseline (기본 하이퍼파라미터)

In [4]:
def build_model(**overrides):
    params = dict(
        random_state=RANDOM_STATE,
        n_jobs=-1,
        tree_method="hist",
        objective="binary:logistic",
        eval_metric="logloss",
    )
    params.update(overrides)
    return XGBClassifier(**params)


def fit_select_evaluate(train_df, valid_df, test_df, feature_columns, model):
    model.fit(train_df[feature_columns], train_df[TARGET])
    valid_proba = model.predict_proba(valid_df[feature_columns])[:, 1]
    threshold = select_threshold(valid_df[TARGET], valid_proba)
    valid_result = evaluate_at_threshold(valid_df[TARGET], valid_proba, threshold)
    test_proba = model.predict_proba(test_df[feature_columns])[:, 1]
    test_result = evaluate_at_threshold(test_df[TARGET], test_proba, threshold)
    return valid_result, test_result


type_splits = {}
for inspection_type in sorted(clean_df["inspection_type"].unique()):
    type_mask = clean_df["inspection_type"] == inspection_type
    type_train_df = clean_df.loc[train_mask & type_mask]
    type_valid_df = clean_df.loc[valid_mask & type_mask]
    type_test_df = clean_df.loc[test_mask & type_mask]
    type_feature_columns = get_non_constant_columns(feature_columns_all, type_train_df)
    type_splits[inspection_type] = {
        "train": type_train_df, "valid": type_valid_df, "test": type_test_df,
        "feature_columns": type_feature_columns,
    }

baseline_test_results = {}
for inspection_type, split in type_splits.items():
    _, test_result = fit_select_evaluate(
        split["train"], split["valid"], split["test"], split["feature_columns"], build_model()
    )
    baseline_test_results[inspection_type] = test_result

baseline_df = pd.DataFrame(baseline_test_results).T
baseline_df.index.name = "inspection_type"
baseline_df[["threshold", "tn", "fp", "fn", "tp", "pr_auc", "slip_rate", "volume_reduction", "total_cost_1:10", "total_cost_1:100"]]


,threshold,tn,fp,fn,tp,pr_auc,slip_rate,volume_reduction,total_cost_1:10,total_cost_1:100
inspection_type,,,,,,,,,,
0,0.000006,8857.0,7794.0,22.0,111.0,0.067265,0.165414,0.531920,8014.0,9994.0
1,0.000025,1869.0,8459.0,8.0,776.0,0.403144,0.010204,0.180964,8539.0,9259.0
2,0.000006,3492.0,14958.0,12.0,691.0,0.356827,0.017070,0.189268,15078.0,16158.0
3,0.000007,9700.0,20288.0,43.0,560.0,0.269568,0.071310,0.323463,20718.0,24588.0
4,0.000001,0.0,730.0,0.0,26.0,0.023284,0.000000,0.000000,730.0,730.0


## 5. Phase A — 하이퍼파라미터 랜덤서치 (리샘플링 없이 baseline 데이터로)

정규화 계열 위주로 탐색 공간을 잡는다: `max_depth`(얕을수록 정규화), `min_child_weight`/`gamma`
(분기를 더 신중하게), `subsample`/`colsample_bytree`(행·열 서브샘플링), `learning_rate`+
`n_estimators`(느리게 배우고 더 많이). 20개 조합을 무작위로 뽑아 5개 유형 공통으로 적용한다.
Validation 총비용(두 시나리오 각각)이 가장 낮은 조합을 유형×시나리오별로 고른다.


In [5]:
rng = np.random.RandomState(RANDOM_STATE)
param_space = {
    "max_depth": [2, 3, 4, 6],
    "min_child_weight": [1, 3, 5, 10, 20],
    "gamma": [0, 0.5, 1, 3, 5],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "learning_rate": [0.03, 0.1, 0.3],
    "n_estimators": [100, 300],
}

N_CANDIDATES = 20
candidates = [{}]  # index 0 = 항상 기본값(빈 dict, 비교 기준) - 아래서 pop 대신 .items()만 읽어 원본 보존
for _ in range(N_CANDIDATES):
    candidates.append({k: rng.choice(v) for k, v in param_space.items()})


def candidate_to_model_params(params):
    return {k: (int(v) if k in ("max_depth", "n_estimators", "min_child_weight") else float(v)) for k, v in params.items()}


phase_a_rows = []
for candidate_idx, params in enumerate(candidates):
    is_default = candidate_idx == 0
    model_params = candidate_to_model_params(params)

    for inspection_type, split in type_splits.items():
        valid_result, test_result = fit_select_evaluate(
            split["train"], split["valid"], split["test"], split["feature_columns"], build_model(**model_params)
        )
        phase_a_rows.append(
            {
                "candidate": candidate_idx, "is_default": is_default, "inspection_type": inspection_type,
                **{f"param_{k}": v for k, v in model_params.items()},
                "valid_total_cost_1:10": valid_result["total_cost_1:10"],
                "valid_total_cost_1:100": valid_result["total_cost_1:100"],
                "test_tn": test_result["tn"], "test_fp": test_result["fp"],
                "test_fn": test_result["fn"], "test_tp": test_result["tp"],
                "test_total_cost_1:10": test_result["total_cost_1:10"],
                "test_total_cost_1:100": test_result["total_cost_1:100"],
            }
        )

phase_a_df = pd.DataFrame(phase_a_rows)
print(f"총 {len(candidates)}개 조합 x {len(type_splits)}개 유형 = {len(phase_a_df)}행")
phase_a_df.head()


총 21개 조합 x 5개 유형 = 105행


,candidate,is_default,inspection_type,valid_total_cost_1:10,valid_total_cost_1:100,test_tn,test_fp,test_fn,test_tp,test_total_cost_1:10,test_total_cost_1:100,param_max_depth,param_min_child_weight,param_gamma,param_subsample,param_colsample_bytree,param_learning_rate,param_n_estimators
0,0,True,0,12559,12559,8857,7794,22,111,8014,9994,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0,True,1,9156,9336,1869,8459,8,776,8539,9259,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0,True,2,13860,13860,3492,14958,12,691,15078,16158,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0,True,3,13831,13831,9700,20288,43,560,20718,24588,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0,True,4,1103,1103,0,730,0,26,730,730,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 6. Phase A 결과 — Validation 기준 유형×시나리오별 최고 하이퍼파라미터, Test에서 확인

In [6]:
phase_a_best_rows = []
for inspection_type in sorted(clean_df["inspection_type"].unique()):
    type_df = phase_a_df[phase_a_df["inspection_type"] == inspection_type]
    default_row = type_df[type_df["is_default"]].iloc[0]

    for scenario in ["1:10", "1:100"]:
        best_row = type_df.loc[type_df[f"valid_total_cost_{scenario}"].idxmin()]
        phase_a_best_rows.append(
            {
                "inspection_type": inspection_type,
                "비용비율": scenario,
                "best_candidate": int(best_row["candidate"]),
                "is_default_best": bool(best_row["is_default"]),
                "baseline_TN": int(default_row["test_tn"]), "baseline_FN": int(default_row["test_fn"]),
                "tuned_TN": int(best_row["test_tn"]), "tuned_FN": int(best_row["test_fn"]),
                "ΔTN": int(best_row["test_tn"] - default_row["test_tn"]),
                "ΔFN": int(best_row["test_fn"] - default_row["test_fn"]),
                "baseline_총비용": default_row[f"test_total_cost_{scenario}"],
                "tuned_총비용": best_row[f"test_total_cost_{scenario}"],
                "개선": best_row[f"test_total_cost_{scenario}"] - default_row[f"test_total_cost_{scenario}"],
            }
        )

phase_a_best_df = pd.DataFrame(phase_a_best_rows).set_index(["inspection_type", "비용비율"])
phase_a_best_df


best_candidate  is_default_best  baseline_TN  \
inspection_type 비용비율                                                  
0               1:10                1            False         8857   
                1:100               1            False         8857   
1               1:10               16            False         1869   
                1:100              16            False         1869   
2               1:10               15            False         3492   
                1:100              15            False         3492   
3               1:10                5            False         9700   
                1:100               5            False         9700   
4               1:10               11            False            0   
                1:100              11            False            0   

                       baseline_FN  tuned_TN  tuned_FN   ΔTN  ΔFN  \
inspection_type 비용비율                                                
0               1:10            22     11013        36  2156   14   
                1:100           22     11013        36  2156   14   
1               1:10             8      4798        30  2929   22   
                1:100            8      4798        30  2929   22   
2               1:10            12      9769        30  6277   18   
                1:100           12      9769        30  6277   18   
3               1:10            43     16182        47  6482    4   
                1:100           43     16182        47  6482    4   
4               1:10             0        13         0    13    0   
                1:100            0        13         0    13    0   

                       baseline_총비용  tuned_총비용    개선  
inspection_type 비용비율                                  
0               1:10           8014       5998 -2016  
                1:100          9994       9238  -756  
1               1:10           8539       5830 -2709  
                1:100          9259       8530  -729  
2               1:10          15078       8981 -6097  
                1:100         16158      11681 -4477  
3               1:10          20718      14276 -6442  
                1:100         24588      18506 -6082  
4               1:10            730        717   -13  
                1:100           730        717   -13

## 7. Phase B — "현재 최고 기법" 데이터 위에서 하이퍼파라미터를 다시 탐색

**1차 시도(실패)**: 처음엔 5절에서 baseline(순수) 데이터로 고른 최적 하이퍼파라미터를 그대로
리샘플링된 데이터에 얹었더니 전 유형에서 참패했다(예: type0은 TN=0, FN=0까지 무너짐). 원인은
명확하다 — 리샘플링(특히 undersample)은 양성:음성 비율을 원본(1:100+)에서 훨씬 완만하게
(1:4 근처로) 바꾸는데, baseline 데이터에 맞춰 고른 정규화 강도(min_child_weight/gamma 등)는
그 극단적 불균형에 맞춘 것이라 다른 비율의 데이터에는 안 맞는다. **하이퍼파라미터와 데이터
전처리는 따로 고를 수 없고 같이 탐색해야 한다.**

그래서 이번엔 각 유형의 "현재 최고 기법"으로 이미 처리된 Train 데이터 위에서 **하이퍼파라미터를
새로 탐색**한다(5절과 같은 후보 21개 재사용, Validation은 항상 원본 그대로).


In [7]:
current_best_technique = {
    "1:10": {0: "label_cleansing", 1: "undersample", 2: "undersample", 3: "cleanlab_smote", 4: "undersample"},
    "1:100": {0: "label_cleansing", 1: "undersample", 2: "undersample", 3: "cleanlab_smote", 4: "undersample"},
}


def apply_technique(train_df, feature_columns, technique):
    if technique == "label_cleansing":
        nunique_classes = train_df.groupby(feature_columns)[TARGET].transform("nunique")
        train_df = train_df.loc[~(nunique_classes > 1)]
        return train_df[feature_columns], train_df[TARGET]
    X_train, y_train = train_df[feature_columns], train_df[TARGET]
    n_pos = int((y_train == 1).sum())
    if technique in ("undersample", "smote", "adasyn"):
        return resample_train(technique, X_train, y_train, n_pos)
    if technique == "cleanlab_smote":
        # cleanlab 이슈 마스크는 007에서만 계산했으므로 여기서는 smote까지만 재현(라벨 정제 부분은
        # 007 노트북 참고) - 이 노트북에서는 하이퍼파라미터 효과만 분리해서 보기 위해 smote로 근사.
        return resample_train("smote", X_train, y_train, n_pos)
    return X_train, y_train


phase_b_rows = []
for scenario in ["1:10", "1:100"]:
    for inspection_type, split in type_splits.items():
        technique = current_best_technique[scenario][inspection_type]
        train_df, valid_df, test_df = split["train"], split["valid"], split["test"]
        feature_columns = split["feature_columns"]

        X_res, y_res = apply_technique(train_df, feature_columns, technique)

        candidate_results = []
        for candidate_idx, params in enumerate(candidates):
            model_params = candidate_to_model_params(params)
            model = build_model(**model_params)
            model.fit(X_res, y_res)
            valid_proba = model.predict_proba(valid_df[feature_columns])[:, 1]
            threshold = select_threshold(valid_df[TARGET], valid_proba)
            valid_result = evaluate_at_threshold(valid_df[TARGET], valid_proba, threshold)
            test_proba = model.predict_proba(test_df[feature_columns])[:, 1]
            test_result = evaluate_at_threshold(test_df[TARGET], test_proba, threshold)
            candidate_results.append(
                {"candidate": candidate_idx, "is_default": candidate_idx == 0,
                 "valid_cost": valid_result[f"total_cost_{scenario}"], "test_result": test_result}
            )

        default_entry = candidate_results[0]
        best_entry = min(candidate_results, key=lambda c: c["valid_cost"])

        base_result = default_entry["test_result"]
        tuned_result = best_entry["test_result"]

        phase_b_rows.append(
            {
                "inspection_type": inspection_type, "비용비율": scenario, "기법": technique,
                "best_candidate": best_entry["candidate"], "is_default_best": best_entry["is_default"],
                "현재최고(기본HP)_TN": base_result["tn"], "현재최고(기본HP)_FN": base_result["fn"],
                "튜닝HP_TN": tuned_result["tn"], "튜닝HP_FN": tuned_result["fn"],
                "ΔTN(현재최고 대비)": tuned_result["tn"] - base_result["tn"],
                "ΔFN(현재최고 대비)": tuned_result["fn"] - base_result["fn"],
                "현재최고_총비용": base_result[f"total_cost_{scenario}"],
                "튜닝HP_총비용": tuned_result[f"total_cost_{scenario}"],
                "개선": tuned_result[f"total_cost_{scenario}"] - base_result[f"total_cost_{scenario}"],
            }
        )

phase_b_df = pd.DataFrame(phase_b_rows).set_index(["inspection_type", "비용비율"])
phase_b_df


,,기법,best_candidate,is_default_best,현재최고(기본HP)_TN,현재최고(기본HP)_FN,튜닝HP_TN,튜닝HP_FN,ΔTN(현재최고 대비),ΔFN(현재최고 대비),현재최고_총비용,튜닝HP_총비용,개선
inspection_type,비용비율,,,,,,,,,,,,
0,1:10,label_cleansing,0,True,11433,46,11433,46,0,0,5678,5678,0
1,1:10,undersample,8,False,4239,12,4815,36,576,24,6209,5873,-336
2,1:10,undersample,1,False,8339,43,12934,80,4595,37,10541,6316,-4225
3,1:10,cleanlab_smote,10,False,10632,31,20575,69,9943,38,19666,10103,-9563
4,1:10,undersample,0,True,15,0,15,0,0,0,715,715,0
0,1:100,label_cleansing,0,True,11433,46,11433,46,0,0,9818,9818,0
1,1:100,undersample,8,False,4239,12,4815,36,576,24,7289,9113,1824
2,1:100,undersample,1,False,8339,43,12934,80,4595,37,14411,13516,-895
3,1:100,cleanlab_smote,10,False,10632,31,20575,69,9943,38,22456,16313,-6143


## 8. 결론 및 다음 단계

### Phase A — baseline 데이터에 하이퍼파라미터만 튜닝 (리샘플링 없음)

| type | baseline 총비용(1:10→1:100) | 튜닝 후(1:10→1:100) | 개선 |
|---|---|---|---:|
| 0 | 8,014 → 9,994 | 5,998 → 9,238 | **-2,016 / -756** |
| 1 | 8,539 → 9,259 | 5,830 → 8,530 | **-2,709 / -729** |
| 2 | 15,078 → 16,158 | 8,981 → 11,681 | **-6,097 / -4,477** |
| 3 | 20,718 → 24,588 | 14,276 → 18,506 | **-6,442 / -6,082** |
| 4 | 730 → 730 | 717 → 717 | -13 / -13 |

**전 유형·전 시나리오에서 예외 없이 개선됐다.** 데이터를 하나도 안 건드리고 정규화 계열
하이퍼파라미터(얕은 트리, 큰 min_child_weight/gamma, 서브샘플링)만 바꿨을 뿐인데, type2/3은
30~40%대 비용 절감이다. 지금까지 반복 관찰된 "임계값이 극단으로 내려간다", "표본 작은 유형이
불안정하다" 같은 증상이 실제로 정규화 부족 때문이었다는 게 확인된 셈이다. **이번 프로젝트에서
가장 크고 일관된 단일 개선**이다.

### Phase B — 각 유형의 "현재 최고 기법" 데이터 위에서 하이퍼파라미터 재탐색

(1차 시도는 Phase A의 챔피언 설정을 그대로 재사용했다가 전 유형에서 참패했다 — 리샘플링으로
양성:음성 비율이 완전히 달라지면 baseline에 맞춘 정규화 강도가 안 맞기 때문. 데이터 위에서
다시 탐색하도록 고쳐서 재실행했다.)

| type | 기법 | 현재최고(기본HP) 1:10→1:100 | 튜닝HP 1:10→1:100 | 개선 |
|---|---|---|---|---:|
| 0 | label_cleansing | 5,678 → 9,818 | 동일(기본이 최적) | 0 / 0 |
| 1 | undersample | 6,209 → 7,289 | 5,873 → 9,113 | **-336** / +1,824(악화) |
| 2 | undersample | 10,541 → 14,411 | 6,316 → 13,516 | **-4,225 / -895** |
| 3 | smote(cleanlab 근사)† | 19,666 → 22,456 | 10,103 → 16,313 | **-9,563 / -6,143** |
| 4 | undersample | 715 → 715 | 동일(기본이 최적) | 0 / 0 |

† type3의 "현재최고"는 007의 실제 cleanlab+smote(1:10=10,058)가 아니라 cleanlab 없이 smote만
적용한 근사치다(cleanlab 이슈 마스크는 이 노트북에서 다시 계산하지 않음) — 하이퍼파라미터
효과만 분리해서 보려고 근사했다. **실제 cleanlab+smote 위에서 재탐색하면 숫자가 달라질 수
있다 — 다음 세션 확인 필요.**

**type1은 흥미로운 경고 사례다**: 1:10에서는 튜닝이 이겼지만(-336), **1:100에서는 오히려
악화됐다(+1,824)** — Validation 기준으로 고른 설정이 Validation에서는 더 좋아 보였지만 Test에서
어긋났다(일반화 실패). 후보 21개짜리 가벼운 랜덤서치도 특정 비용 시나리오에 과적합할 수 있다는
뜻이라, 최종 채택 전 반드시 두 시나리오 모두에서 확인해야 한다.

### 검사유형별 최적 조합 갱신 (003~009 종합)

| type | 최종 기법 | 최종 하이퍼파라미터 | 비고 |
|---|---|---|---|
| 0 | label_cleansing | 기본값 유지 | 표본 부족, 튜닝 무의미 |
| 1 | undersample | **1:10만 튜닝 채택**, 1:100은 기본값 유지 | 시나리오별로 다르게 채택 |
| 2 | undersample | **튜닝 채택**(두 시나리오 모두) | 가장 큰 수혜 유형 중 하나 |
| 3 | cleanlab+smote | **튜닝 채택 가능성 높음**(정확한 재확인 필요) | 근사치 기준으로는 최대 수혜 |
| 4 | undersample | 기본값 유지 | 표본 부족 |

### 다음 단계

1. type3을 실제 cleanlab 이슈 마스크 위에서 하이퍼파라미터 재탐색해 정확한 수치 확정.
2. 이번에 찾은 튜닝 설정들을 `docs/model_val.md`에 반영할지 검토(팀 공용 문서).
3. 이상치 탐지 재구현 또는 Phase 4(비지도 이상탐지)로 이동.
